# 07 · Task description and timing (Supplementary Tables S1 and S2)

Task parameters that can be read from the data of the 135 analyzed participants (all 120 trials of each):
task version, first block, response keys, words, accuracy, reaction times on correct and error trials,
task duration, and the timing of successive trials from the time stamps.

* *MATLAB:* the next word appeared 230 ms after the response that ended the trial. This interval follows the
  response and is not part of the RT. An error trial ended only when the correct key was pressed, so the RT of an
  error trial runs to the corrective response.
* *Browser:* the next word appeared immediately. The RT of an error trial is the latency of the first (incorrect)
  response.

In [ ]:
import sys
sys.path.insert(0, "..")          # dbiat_analysis.py is in the folder above notebooks/
import pandas as pd
import dbiat_analysis as A

cfg = A.load_settings()
data = A.load_data(cfg).sort_values(["participant", "block_number", "trial_in_block"])
pd.set_option("display.width", 250, "display.max_colwidth", 140)

## 1. Timing from the time stamps

In [ ]:
# MATLAB: interval between successive word onsets within a block, minus the RT
ml = data[data.task_version == "MATLAB"].copy()
nxt = ml.groupby(["participant", "block_number"]).stimulus_onset_s.shift(-1)
ml["rest_ms"] = (nxt - ml.stimulus_onset_s) * 1000 - ml.rt_ms
ml_rest = ml.groupby("correct").rest_ms.median()
ml_iti, ml_lo, ml_hi = ml.rest_ms.median(), ml.rest_ms.quantile(.025), ml.rest_ms.quantile(.975)

# Browser: time_elapsed is written when a trial ends, so (end of this trial − end of the previous trial − RT) is the
# time from the first response to the end of the trial (0 when the first response was correct).
# The first trial of each block follows an instruction screen and is skipped.
js = data[data.task_version == "browser"].copy()
js["extra_ms"] = js.time_elapsed_ms - js.groupby(["participant", "block_number"]).time_elapsed_ms.shift(1) - js.rt_ms
js_extra = js.groupby("correct").extra_ms.median()
js_q99 = js.loc[js.correct == 1, "extra_ms"].quantile(.99)

timing = pd.DataFrame([
    dict(version="MATLAB", quantity="onset interval − RT, correct trials (median, ms)", value=ml_rest[1]),
    dict(version="MATLAB", quantity="onset interval − RT, error trials (median, ms)", value=ml_rest[0]),
    dict(version="MATLAB", quantity="onset interval − RT, all trials: 2.5th percentile (ms)", value=ml_lo),
    dict(version="MATLAB", quantity="onset interval − RT, all trials: 97.5th percentile (ms)", value=ml_hi),
    dict(version="browser", quantity="first response to end of trial, correct trials (99th percentile, ms)", value=js_q99),
    dict(version="browser", quantity="first response to end of trial, error trials (median, ms)", value=js_extra[0]),
])
A.write_table(timing.round(1), cfg, "task_timing")
timing.round(1)

## 2. Table S1 · Task parameters (values from the data)

In [ ]:
first = data[(data.block_number == 1) & (data.trial_in_block == 1)]
n_ver = first.task_version.value_counts()
dur_js = js.groupby("participant").time_elapsed_ms.agg(lambda x: (x.max() - x.min()) / 60000)
dur_ml = ml.groupby("participant").stimulus_onset_s.agg(lambda x: (x.max() - x.min()) / 60)
acc = data.groupby("task_version").correct.mean()
med = data.groupby(["task_version", "correct"]).rt_ms.median()
keys = "; ".join(f"{v}: " + ", ".join(f"{k} {c}" for k, c in d.target_key.value_counts().sort_index().items())
                 for v, d in first.groupby("task_version"))
s1 = pd.DataFrame([
    ("Participants per task version", f"browser {n_ver['browser']}, MATLAB {n_ver['MATLAB']}"),
    ("First block", f"Life:Me {int((first.block_type == A.LIFE).sum())}, Death:Me {int((first.block_type == A.DEATH).sum())}"),
    ("Target key in block 1", keys),
    ("Median RT, correct / error trials (ms)",
     f"browser {med[('browser', 1)]:.0f} / {med[('browser', 0)]:.0f}; MATLAB {med[('MATLAB', 1)]:.0f} / {med[('MATLAB', 0)]:.0f}"),
    ("Browser: first response to end of error trial (median, ms)", f"{js_extra[0]:.0f}"),
    ("Browser: next word after the response (99% of correct trials, ms)", f"within {js_q99:.0f}"),
    ("MATLAB: next word after the response (median; 95% range, ms)", f"{ml_iti:.0f}; {ml_lo:.0f}–{ml_hi:.0f}"),
    ("MATLAB: onset interval − RT on correct / error trials (median, ms)", f"{ml_rest[1]:.0f} / {ml_rest[0]:.0f}"),
    ("Task duration, first to last trial (median; range, min)",
     f"browser {dur_js.median():.1f}; {dur_js.min():.1f}–{dur_js.max():.1f}. "
     f"MATLAB {dur_ml.median():.1f}; {dur_ml.min():.1f}–{dur_ml.max():.1f}"),
    ("Overall accuracy (all trials)", f"browser {acc['browser']:.1%}; MATLAB {acc['MATLAB']:.1%}"),
], columns=["Parameter", "Value"])
A.write_table(s1, cfg, "TableS1_values")
s1

## 3. Table S2 · Words
Mean number of presentations per participant.

In [ ]:
n_part = data.participant.nunique()
s2 = (data.groupby(["word_category", "word"]).size().rename("presentations").reset_index()
      .assign(per_participant=lambda d: d.presentations / n_part))
s2 = (s2.groupby("word_category")
      .apply(lambda d: ", ".join(f"{w} ({p:.1f})" for w, p in zip(d.word, d.per_participant)), include_groups=False)
      .rename("words (mean presentations per participant)").reindex(["Me", "Life", "Death", "Other"]).to_frame())
per_block = data.groupby(["participant", "block_number", "word_category"]).size().groupby("word_category").agg(["min", "max"])
s2 = s2.join(per_block.rename(columns={"min": "per_block_min", "max": "per_block_max"}))
A.write_table(s2.reset_index().rename(columns={"word_category": "category"}), cfg, "TableS2_words")
s2